In [ ]:
import os
import shutil
import json
import re
from collections import defaultdict

# ========== 配置区（按需修改） ==========
base_path = r"C:\Users\Admin\Documents\SynologyDrive\0-data\H-D_splitting\Raw Data\4-manual\robotic_arm\ML"
destination_path = r"C:\Users\Admin\Documents\GitHub\hd_separation_ml\data_extraction\all_data\EIS"
# ========================================

os.makedirs(destination_path, exist_ok=True)

def normalize(text):
    if text is None:
        return ""
    s = str(text).strip()
    # 去掉扩展名 / json 后缀，转换小写，统一一些符号
    s = re.sub(r'\.json$', '', s, flags=re.IGNORECASE)
    s = s.replace(' ', '_')
    s = s.lower()
    return s

def container_to_list(container):
    """
    把可能的 JSON 字段（list / dict / 单值）统一成 list。
    dict 会尝试按数字键排序（若键都是数字），否则按原 dict 顺序返回 values 列表。
    """
    if container is None:
        return []
    if isinstance(container, list):
        return container
    if isinstance(container, dict):
        # 尝试把数字键按数值排序
        keys = list(container.keys())
        try:
            # 若所有 key 可转 int，则按 int 排
            if all(re.fullmatch(r'-?\d+', str(k)) for k in keys):
                keys_sorted = sorted(keys, key=lambda k: int(k))
                return [container[k] for k in keys_sorted]
        except Exception:
            pass
        # fallback: values 顺序
        return list(container.values())
    # 单一值
    return [container]

def build_combos(data):
    s1 = container_to_list(data.get('substance1'))
    s2 = container_to_list(data.get('substance2'))
    n = max(len(s1), len(s2))
    # pad short lists with None
    s1 += [None] * (n - len(s1))
    s2 += [None] * (n - len(s2))
    combos = []
    for i in range(n):
        combos.append({
            'index': i,
            'sub1_raw': s1[i],
            'sub2_raw': s2[i],
            'sub1_norm': normalize(s1[i]),
            'sub2_norm': normalize(s2[i]),
            'label_raw': f"{s1[i]}-{s2[i]}" if s1[i] is not None or s2[i] is not None else None,
            'label_norm': normalize(f"{s1[i]}-{s2[i]}") if (s1[i] or s2[i]) else ""
        })
    return combos

def find_best_match_for_basename(basename_norm, basename_raw, combos):
    """
    尝试多策略匹配，返回匹配到的 index 或 None，以及匹配理由字符串
    策略顺序：
      1) 文件名尾部数字 -> 当作 index（支持 0-based 或 1-based 尝试）
      2) 精确 label match（sub1-sub2）
      3) 精确 sub1 或 sub2 匹配（若唯一则采用）
      4) 部分匹配（basename 包含在 label 中或 label 包含 basename）
      5) 无匹配 -> None
    """
    # 1) 尝试尾部数字
    m = re.search(r'(\d+)$', basename_raw)
    if m:
        num = int(m.group(1))
        # 尝试 0-based
        if 0 <= num < len(combos):
            return num, f"numeric_suffix_matched_0based({num})"
        # 尝试 1-based
        if 1 <= num <= len(combos):
            idx = num - 1
            return idx, f"numeric_suffix_matched_1based({num} -> {idx})"

    # 2) 精确 label match（sub1-sub2）
    for c in combos:
        if basename_norm and c['label_norm'] and basename_norm == c['label_norm']:
            return c['index'], "exact_label_match"

    # 3) 精确 sub1 或 sub2 唯一匹配
    sub1_matches = [c['index'] for c in combos if basename_norm and basename_norm == c['sub1_norm']]
    sub2_matches = [c['index'] for c in combos if basename_norm and basename_norm == c['sub2_norm']]
    if len(sub1_matches) == 1:
        return sub1_matches[0], "unique_sub1_match"
    if len(sub2_matches) == 1:
        return sub2_matches[0], "unique_sub2_match"

    # 4) 部分匹配（包含关系）
    partial = []
    for c in combos:
        if basename_norm and (basename_norm in c['label_norm'] or c['label_norm'] in basename_norm or basename_norm in c['sub1_norm'] or basename_norm in c['sub2_norm']):
            partial.append(c['index'])
    if len(partial) == 1:
        return partial[0], "partial_single_match"
    if len(partial) > 1:
        return partial[0], f"partial_multiple_match_choose_first({partial})"  # 可改为返回 None

    # 无匹配
    return None, "no_match"

# 主流程：遍历 base_path 下每个子文件夹
summary = []  # 用于写 mapping_summary.json
for folder in os.listdir(base_path):
    folder_path = os.path.join(base_path, folder)
    if not os.path.isdir(folder_path):
        continue

    tmp_json_path = os.path.join(folder_path, "tmp", "initial_sampling.json")
    if not os.path.exists(tmp_json_path):
        print(f"[WARN] JSON not found: {tmp_json_path}  (跳过)")
        continue

    try:
        with open(tmp_json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
    except Exception as e:
        print(f"[ERROR] Failed to load {tmp_json_path}: {e}")
        continue

    combos = build_combos(data)
    if not combos:
        print(f"[WARN] {tmp_json_path} 中没有 substance1/substance2 内容 (跳过)")
        continue

    EIS_path = os.path.join(folder_path, "tmp", "electrocatalysis", "EIS")
    if not os.path.isdir(EIS_path):
        print(f"[WARN] EIS folder not found: {EIS_path} (跳过)")
        continue

    txt_files = sorted([f for f in os.listdir(EIS_path) if f.lower().endswith('.txt')])
    assigned_indices = {}   # index -> target info
    file_reports = []

    for fname in txt_files:
        base_raw = os.path.splitext(fname)[0]
        base_norm = normalize(base_raw)

        matched_idx, reason = find_best_match_for_basename(base_norm, base_raw, combos)

        if matched_idx is None:
            # 回退策略：按尚未填充的最小索引顺序分配
            unassigned = [c['index'] for c in combos if c['index'] not in assigned_indices]
            if unassigned:
                matched_idx = unassigned[0]
                reason = f"fallback_sequential_assigned({matched_idx})"
            else:
                reason = "fallback_no_unassigned_index"
                matched_idx = None

        target_name = None
        if matched_idx is not None:
            c = combos[matched_idx]
            s1 = c['sub1_raw'] if c['sub1_raw'] is not None else "UnknownSub1"
            s2 = c['sub2_raw'] if c['sub2_raw'] is not None else "UnknownSub2"
            # 生成安全的文件名（把可能的斜杠去掉）
            safe_s1 = re.sub(r'[\\/:*?"<>|]', '_', str(s1))
            safe_s2 = re.sub(r'[\\/:*?"<>|]', '_', str(s2))
            target_name = f"{safe_s1}-{safe_s2}.txt"
            # 若已存在同名（可能不同原文件匹配到同一 index），加上索引避免覆盖
            target_path = os.path.join(destination_path, target_name)
            if os.path.exists(target_path):
                target_name = f"{safe_s1}-{safe_s2}.txt"
            assigned_indices[matched_idx] = {
                'txt_source': os.path.join(EIS_path, fname),
                'target_name': target_name,
                'match_reason': reason
            }
            # 复制文件
            try:
                shutil.copy(os.path.join(EIS_path, fname), os.path.join(destination_path, target_name))
            except Exception as e:
                print(f"[ERROR] Copy failed for {fname}: {e}")
        else:
            # 无法匹配到任何 index，保留原名并在文件名加上 folder 前缀防止冲突
            target_name = f"{folder}_{base_raw}.txt"
            try:
                shutil.copy(os.path.join(EIS_path, fname), os.path.join(destination_path, target_name))
            except Exception as e:
                print(f"[ERROR] Copy fallback failed for {fname}: {e}")

        file_reports.append({
            'source_file': fname,
            'base_raw': base_raw,
            'base_norm': base_norm,
            'matched_index': matched_idx,
            'target_name': target_name,
            'reason': reason
        })
        print(f"[INFO] {fname} -> matched_index={matched_idx} reason={reason} -> {target_name}")

    # 记录哪些 index 没有被任何文件匹配到
    missing_indices = [c['index'] for c in combos if c['index'] not in assigned_indices]

    summary.append({
        'source_folder': folder,
        'json_path': tmp_json_path,
        'EIS_path': EIS_path,
        'num_json_entries': len(combos),
        'num_txt_files_found': len(txt_files),
        'assigned_count': len(assigned_indices),
        'missing_indices': missing_indices,
        'file_reports': file_reports
    })

# 写 summary 报告
summary_path = os.path.join(destination_path, "mapping_summary.json")
with open(summary_path, 'w', encoding='utf-8') as outf:
    json.dump(summary, outf, ensure_ascii=False, indent=2)

print(f"Done. Mapping summary saved to: {summary_path}")


[INFO] 0.txt -> matched_index=0 reason=numeric_suffix_matched_0based(0) -> Eu-Co.txt
[INFO] 1.txt -> matched_index=1 reason=numeric_suffix_matched_0based(1) -> Y-Co.txt
[INFO] 10.txt -> matched_index=10 reason=numeric_suffix_matched_0based(10) -> Cr-Co.txt
[INFO] 11.txt -> matched_index=11 reason=numeric_suffix_matched_0based(11) -> In-Co.txt
[INFO] 2.txt -> matched_index=2 reason=numeric_suffix_matched_0based(2) -> V-Co.txt
[INFO] 3.txt -> matched_index=3 reason=numeric_suffix_matched_0based(3) -> Ga-Co.txt
[INFO] 4.txt -> matched_index=4 reason=numeric_suffix_matched_0based(4) -> W-Co.txt
[INFO] 5.txt -> matched_index=5 reason=numeric_suffix_matched_0based(5) -> Mn-Co.txt
[INFO] 6.txt -> matched_index=6 reason=numeric_suffix_matched_0based(6) -> Zn-Co.txt
[INFO] 7.txt -> matched_index=7 reason=numeric_suffix_matched_0based(7) -> Cd-Co.txt
[INFO] 8.txt -> matched_index=8 reason=numeric_suffix_matched_0based(8) -> La-Co.txt
[INFO] 9.txt -> matched_index=9 reason=numeric_suffix_matched_

In [ ]:
import os
import shutil
import json
import re
from collections import defaultdict

# ========== 配置区（按需修改） ==========
base_path = r"C:\Users\Admin\Documents\SynologyDrive\0-data\H-D_splitting\Raw Data\4-manual\robotic_arm\ML"
destination_path = r"C:\Users\Admin\Documents\GitHub\hd_separation_ml\data_extraction\all_data\LSV"
# ========================================

os.makedirs(destination_path, exist_ok=True)

def normalize(text):
    if text is None:
        return ""
    s = str(text).strip()
    # 去掉扩展名 / json 后缀，转换小写，统一一些符号
    s = re.sub(r'\.json$', '', s, flags=re.IGNORECASE)
    s = s.replace(' ', '_')
    s = s.lower()
    return s

def container_to_list(container):
    """
    把可能的 JSON 字段（list / dict / 单值）统一成 list。
    dict 会尝试按数字键排序（若键都是数字），否则按原 dict 顺序返回 values 列表。
    """
    if container is None:
        return []
    if isinstance(container, list):
        return container
    if isinstance(container, dict):
        # 尝试把数字键按数值排序
        keys = list(container.keys())
        try:
            # 若所有 key 可转 int，则按 int 排
            if all(re.fullmatch(r'-?\d+', str(k)) for k in keys):
                keys_sorted = sorted(keys, key=lambda k: int(k))
                return [container[k] for k in keys_sorted]
        except Exception:
            pass
        # fallback: values 顺序
        return list(container.values())
    # 单一值
    return [container]

def build_combos(data):
    s1 = container_to_list(data.get('substance1'))
    s2 = container_to_list(data.get('substance2'))
    n = max(len(s1), len(s2))
    # pad short lists with None
    s1 += [None] * (n - len(s1))
    s2 += [None] * (n - len(s2))
    combos = []
    for i in range(n):
        combos.append({
            'index': i,
            'sub1_raw': s1[i],
            'sub2_raw': s2[i],
            'sub1_norm': normalize(s1[i]),
            'sub2_norm': normalize(s2[i]),
            'label_raw': f"{s1[i]}-{s2[i]}" if s1[i] is not None or s2[i] is not None else None,
            'label_norm': normalize(f"{s1[i]}-{s2[i]}") if (s1[i] or s2[i]) else ""
        })
    return combos

def find_best_match_for_basename(basename_norm, basename_raw, combos):
    """
    尝试多策略匹配，返回匹配到的 index 或 None，以及匹配理由字符串
    策略顺序：
      1) 文件名尾部数字 -> 当作 index（支持 0-based 或 1-based 尝试）
      2) 精确 label match（sub1-sub2）
      3) 精确 sub1 或 sub2 匹配（若唯一则采用）
      4) 部分匹配（basename 包含在 label 中或 label 包含 basename）
      5) 无匹配 -> None
    """
    # 1) 尝试尾部数字
    m = re.search(r'(\d+)$', basename_raw)
    if m:
        num = int(m.group(1))
        # 尝试 0-based
        if 0 <= num < len(combos):
            return num, f"numeric_suffix_matched_0based({num})"
        # 尝试 1-based
        if 1 <= num <= len(combos):
            idx = num - 1
            return idx, f"numeric_suffix_matched_1based({num} -> {idx})"

    # 2) 精确 label match（sub1-sub2）
    for c in combos:
        if basename_norm and c['label_norm'] and basename_norm == c['label_norm']:
            return c['index'], "exact_label_match"

    # 3) 精确 sub1 或 sub2 唯一匹配
    sub1_matches = [c['index'] for c in combos if basename_norm and basename_norm == c['sub1_norm']]
    sub2_matches = [c['index'] for c in combos if basename_norm and basename_norm == c['sub2_norm']]
    if len(sub1_matches) == 1:
        return sub1_matches[0], "unique_sub1_match"
    if len(sub2_matches) == 1:
        return sub2_matches[0], "unique_sub2_match"

    # 4) 部分匹配（包含关系）
    partial = []
    for c in combos:
        if basename_norm and (basename_norm in c['label_norm'] or c['label_norm'] in basename_norm or basename_norm in c['sub1_norm'] or basename_norm in c['sub2_norm']):
            partial.append(c['index'])
    if len(partial) == 1:
        return partial[0], "partial_single_match"
    if len(partial) > 1:
        return partial[0], f"partial_multiple_match_choose_first({partial})"  # 可改为返回 None

    # 无匹配
    return None, "no_match"

# 主流程：遍历 base_path 下每个子文件夹
summary = []  # 用于写 mapping_summary.json
for folder in os.listdir(base_path):
    folder_path = os.path.join(base_path, folder)
    if not os.path.isdir(folder_path):
        continue

    tmp_json_path = os.path.join(folder_path, "tmp", "initial_sampling.json")
    if not os.path.exists(tmp_json_path):
        print(f"[WARN] JSON not found: {tmp_json_path}  (跳过)")
        continue

    try:
        with open(tmp_json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
    except Exception as e:
        print(f"[ERROR] Failed to load {tmp_json_path}: {e}")
        continue

    combos = build_combos(data)
    if not combos:
        print(f"[WARN] {tmp_json_path} 中没有 substance1/substance2 内容 (跳过)")
        continue

    EIS_path = os.path.join(folder_path, "tmp", "electrocatalysis\LSV", "2")
    if not os.path.isdir(EIS_path):
        print(f"[WARN] EIS folder not found: {EIS_path} (跳过)")
        continue

    txt_files = sorted([f for f in os.listdir(EIS_path) if f.lower().endswith('.txt')])
    assigned_indices = {}   # index -> target info
    file_reports = []

    for fname in txt_files:
        base_raw = os.path.splitext(fname)[0]
        base_norm = normalize(base_raw)

        matched_idx, reason = find_best_match_for_basename(base_norm, base_raw, combos)

        if matched_idx is None:
            # 回退策略：按尚未填充的最小索引顺序分配
            unassigned = [c['index'] for c in combos if c['index'] not in assigned_indices]
            if unassigned:
                matched_idx = unassigned[0]
                reason = f"fallback_sequential_assigned({matched_idx})"
            else:
                reason = "fallback_no_unassigned_index"
                matched_idx = None

        target_name = None
        if matched_idx is not None:
            c = combos[matched_idx]
            s1 = c['sub1_raw'] if c['sub1_raw'] is not None else "UnknownSub1"
            s2 = c['sub2_raw'] if c['sub2_raw'] is not None else "UnknownSub2"
            # 生成安全的文件名（把可能的斜杠去掉）
            safe_s1 = re.sub(r'[\\/:*?"<>|]', '_', str(s1))
            safe_s2 = re.sub(r'[\\/:*?"<>|]', '_', str(s2))
            target_name = f"{safe_s1}-{safe_s2}.txt"
            # 若已存在同名（可能不同原文件匹配到同一 index），加上索引避免覆盖
            target_path = os.path.join(destination_path, target_name)
            if os.path.exists(target_path):
                target_name = f"{safe_s1}-{safe_s2}.txt"
            assigned_indices[matched_idx] = {
                'txt_source': os.path.join(EIS_path, fname),
                'target_name': target_name,
                'match_reason': reason
            }
            # 复制文件
            try:
                shutil.copy(os.path.join(EIS_path, fname), os.path.join(destination_path, target_name))
            except Exception as e:
                print(f"[ERROR] Copy failed for {fname}: {e}")
        else:
            # 无法匹配到任何 index，保留原名并在文件名加上 folder 前缀防止冲突
            target_name = f"{folder}_{base_raw}.txt"
            try:
                shutil.copy(os.path.join(EIS_path, fname), os.path.join(destination_path, target_name))
            except Exception as e:
                print(f"[ERROR] Copy fallback failed for {fname}: {e}")

        file_reports.append({
            'source_file': fname,
            'base_raw': base_raw,
            'base_norm': base_norm,
            'matched_index': matched_idx,
            'target_name': target_name,
            'reason': reason
        })
        print(f"[INFO] {fname} -> matched_index={matched_idx} reason={reason} -> {target_name}")

    # 记录哪些 index 没有被任何文件匹配到
    missing_indices = [c['index'] for c in combos if c['index'] not in assigned_indices]

    summary.append({
        'source_folder': folder,
        'json_path': tmp_json_path,
        'EIS_path': EIS_path,
        'num_json_entries': len(combos),
        'num_txt_files_found': len(txt_files),
        'assigned_count': len(assigned_indices),
        'missing_indices': missing_indices,
        'file_reports': file_reports
    })

# 写 summary 报告
summary_path = os.path.join(destination_path, "mapping_summary-1.json")
with open(summary_path, 'w', encoding='utf-8') as outf:
    json.dump(summary, outf, ensure_ascii=False, indent=2)

print(f"Done. Mapping summary saved to: {summary_path}")

[WARN] EIS folder not found: C:\Users\Admin\Documents\SynologyDrive\0-data\H-D_splitting\Raw Data\4-manual\robotic_arm\ML\0-Co\tmp\electrocatalysis\LSV\2 (跳过)
[WARN] EIS folder not found: C:\Users\Admin\Documents\SynologyDrive\0-data\H-D_splitting\Raw Data\4-manual\robotic_arm\ML\1-Nb\tmp\electrocatalysis\LSV\2 (跳过)
[WARN] EIS folder not found: C:\Users\Admin\Documents\SynologyDrive\0-data\H-D_splitting\Raw Data\4-manual\robotic_arm\ML\10-Sc\tmp\electrocatalysis\LSV\2 (跳过)
[WARN] EIS folder not found: C:\Users\Admin\Documents\SynologyDrive\0-data\H-D_splitting\Raw Data\4-manual\robotic_arm\ML\11-Co-1\tmp\electrocatalysis\LSV\2 (跳过)
[WARN] EIS folder not found: C:\Users\Admin\Documents\SynologyDrive\0-data\H-D_splitting\Raw Data\4-manual\robotic_arm\ML\12-Ni-1\tmp\electrocatalysis\LSV\2 (跳过)
[WARN] EIS folder not found: C:\Users\Admin\Documents\SynologyDrive\0-data\H-D_splitting\Raw Data\4-manual\robotic_arm\ML\13-Fe-1\tmp\electrocatalysis\LSV\2 (跳过)
[WARN] EIS folder not found: C:\User

In [ ]:
import os
import shutil
import json
import re
from collections import defaultdict

# ========== 配置区（按需修改） ==========
base_path = r"C:\Users\Admin\Documents\SynologyDrive\0-data\H-D_splitting\Raw Data\4-manual\robotic_arm\ML"
destination_path = r"C:\Users\Admin\Documents\GitHub\hd_separation_ml\data_extraction\all_data\Et"
# ========================================

os.makedirs(destination_path, exist_ok=True)

def normalize(text):
    if text is None:
        return ""
    s = str(text).strip()
    # 去掉扩展名 / json 后缀，转换小写，统一一些符号
    s = re.sub(r'\.json$', '', s, flags=re.IGNORECASE)
    s = s.replace(' ', '_')
    s = s.lower()
    return s

def container_to_list(container):
    """
    把可能的 JSON 字段（list / dict / 单值）统一成 list。
    dict 会尝试按数字键排序（若键都是数字），否则按原 dict 顺序返回 values 列表。
    """
    if container is None:
        return []
    if isinstance(container, list):
        return container
    if isinstance(container, dict):
        # 尝试把数字键按数值排序
        keys = list(container.keys())
        try:
            # 若所有 key 可转 int，则按 int 排
            if all(re.fullmatch(r'-?\d+', str(k)) for k in keys):
                keys_sorted = sorted(keys, key=lambda k: int(k))
                return [container[k] for k in keys_sorted]
        except Exception:
            pass
        # fallback: values 顺序
        return list(container.values())
    # 单一值
    return [container]

def build_combos(data):
    s1 = container_to_list(data.get('substance1'))
    s2 = container_to_list(data.get('substance2'))
    n = max(len(s1), len(s2))
    # pad short lists with None
    s1 += [None] * (n - len(s1))
    s2 += [None] * (n - len(s2))
    combos = []
    for i in range(n):
        combos.append({
            'index': i,
            'sub1_raw': s1[i],
            'sub2_raw': s2[i],
            'sub1_norm': normalize(s1[i]),
            'sub2_norm': normalize(s2[i]),
            'label_raw': f"{s1[i]}-{s2[i]}" if s1[i] is not None or s2[i] is not None else None,
            'label_norm': normalize(f"{s1[i]}-{s2[i]}") if (s1[i] or s2[i]) else ""
        })
    return combos

def find_best_match_for_basename(basename_norm, basename_raw, combos):
    """
    尝试多策略匹配，返回匹配到的 index 或 None，以及匹配理由字符串
    策略顺序：
      1) 文件名尾部数字 -> 当作 index（支持 0-based 或 1-based 尝试）
      2) 精确 label match（sub1-sub2）
      3) 精确 sub1 或 sub2 匹配（若唯一则采用）
      4) 部分匹配（basename 包含在 label 中或 label 包含 basename）
      5) 无匹配 -> None
    """
    # 1) 尝试尾部数字
    m = re.search(r'(\d+)$', basename_raw)
    if m:
        num = int(m.group(1))
        # 尝试 0-based
        if 0 <= num < len(combos):
            return num, f"numeric_suffix_matched_0based({num})"
        # 尝试 1-based
        if 1 <= num <= len(combos):
            idx = num - 1
            return idx, f"numeric_suffix_matched_1based({num} -> {idx})"

    # 2) 精确 label match（sub1-sub2）
    for c in combos:
        if basename_norm and c['label_norm'] and basename_norm == c['label_norm']:
            return c['index'], "exact_label_match"

    # 3) 精确 sub1 或 sub2 唯一匹配
    sub1_matches = [c['index'] for c in combos if basename_norm and basename_norm == c['sub1_norm']]
    sub2_matches = [c['index'] for c in combos if basename_norm and basename_norm == c['sub2_norm']]
    if len(sub1_matches) == 1:
        return sub1_matches[0], "unique_sub1_match"
    if len(sub2_matches) == 1:
        return sub2_matches[0], "unique_sub2_match"

    # 4) 部分匹配（包含关系）
    partial = []
    for c in combos:
        if basename_norm and (basename_norm in c['label_norm'] or c['label_norm'] in basename_norm or basename_norm in c['sub1_norm'] or basename_norm in c['sub2_norm']):
            partial.append(c['index'])
    if len(partial) == 1:
        return partial[0], "partial_single_match"
    if len(partial) > 1:
        return partial[0], f"partial_multiple_match_choose_first({partial})"  # 可改为返回 None

    # 无匹配
    return None, "no_match"

# 主流程：遍历 base_path 下每个子文件夹
summary = []  # 用于写 mapping_summary.json
for folder in os.listdir(base_path):
    folder_path = os.path.join(base_path, folder)
    if not os.path.isdir(folder_path):
        continue

    tmp_json_path = os.path.join(folder_path, "tmp", "initial_sampling.json")
    if not os.path.exists(tmp_json_path):
        print(f"[WARN] JSON not found: {tmp_json_path}  (跳过)")
        continue

    try:
        with open(tmp_json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
    except Exception as e:
        print(f"[ERROR] Failed to load {tmp_json_path}: {e}")
        continue

    combos = build_combos(data)
    if not combos:
        print(f"[WARN] {tmp_json_path} 中没有 substance1/substance2 内容 (跳过)")
        continue

    EIS_path = os.path.join(folder_path, "tmp", "electrocatalysis", "Et")
    if not os.path.isdir(EIS_path):
        print(f"[WARN] EIS folder not found: {EIS_path} (跳过)")
        continue

    txt_files = sorted([f for f in os.listdir(EIS_path) if f.lower().endswith('.txt')])
    assigned_indices = {}   # index -> target info
    file_reports = []

    for fname in txt_files:
        base_raw = os.path.splitext(fname)[0]
        base_norm = normalize(base_raw)

        matched_idx, reason = find_best_match_for_basename(base_norm, base_raw, combos)

        if matched_idx is None:
            # 回退策略：按尚未填充的最小索引顺序分配
            unassigned = [c['index'] for c in combos if c['index'] not in assigned_indices]
            if unassigned:
                matched_idx = unassigned[0]
                reason = f"fallback_sequential_assigned({matched_idx})"
            else:
                reason = "fallback_no_unassigned_index"
                matched_idx = None

        target_name = None
        if matched_idx is not None:
            c = combos[matched_idx]
            s1 = c['sub1_raw'] if c['sub1_raw'] is not None else "UnknownSub1"
            s2 = c['sub2_raw'] if c['sub2_raw'] is not None else "UnknownSub2"
            # 生成安全的文件名（把可能的斜杠去掉）
            safe_s1 = re.sub(r'[\\/:*?"<>|]', '_', str(s1))
            safe_s2 = re.sub(r'[\\/:*?"<>|]', '_', str(s2))
            target_name = f"{safe_s1}-{safe_s2}.txt"
            # 若已存在同名（可能不同原文件匹配到同一 index），加上索引避免覆盖
            target_path = os.path.join(destination_path, target_name)
            if os.path.exists(target_path):
                target_name = f"{safe_s1}-{safe_s2}.txt"
            assigned_indices[matched_idx] = {
                'txt_source': os.path.join(EIS_path, fname),
                'target_name': target_name,
                'match_reason': reason
            }
            # 复制文件
            try:
                shutil.copy(os.path.join(EIS_path, fname), os.path.join(destination_path, target_name))
            except Exception as e:
                print(f"[ERROR] Copy failed for {fname}: {e}")
        else:
            # 无法匹配到任何 index，保留原名并在文件名加上 folder 前缀防止冲突
            target_name = f"{folder}_{base_raw}.txt"
            try:
                shutil.copy(os.path.join(EIS_path, fname), os.path.join(destination_path, target_name))
            except Exception as e:
                print(f"[ERROR] Copy fallback failed for {fname}: {e}")

        file_reports.append({
            'source_file': fname,
            'base_raw': base_raw,
            'base_norm': base_norm,
            'matched_index': matched_idx,
            'target_name': target_name,
            'reason': reason
        })
        print(f"[INFO] {fname} -> matched_index={matched_idx} reason={reason} -> {target_name}")

    # 记录哪些 index 没有被任何文件匹配到
    missing_indices = [c['index'] for c in combos if c['index'] not in assigned_indices]

    summary.append({
        'source_folder': folder,
        'json_path': tmp_json_path,
        'EIS_path': EIS_path,
        'num_json_entries': len(combos),
        'num_txt_files_found': len(txt_files),
        'assigned_count': len(assigned_indices),
        'missing_indices': missing_indices,
        'file_reports': file_reports
    })

# 写 summary 报告
summary_path = os.path.join(destination_path, "mapping_summary-1.json")
with open(summary_path, 'w', encoding='utf-8') as outf:
    json.dump(summary, outf, ensure_ascii=False, indent=2)

print(f"Done. Mapping summary saved to: {summary_path}")

[INFO] 0.txt -> matched_index=0 reason=numeric_suffix_matched_0based(0) -> Eu-Co.txt
[INFO] 1.txt -> matched_index=1 reason=numeric_suffix_matched_0based(1) -> Y-Co.txt
[INFO] 10.txt -> matched_index=10 reason=numeric_suffix_matched_0based(10) -> Cr-Co.txt
[INFO] 11.txt -> matched_index=11 reason=numeric_suffix_matched_0based(11) -> In-Co.txt
[INFO] 2.txt -> matched_index=2 reason=numeric_suffix_matched_0based(2) -> V-Co.txt
[INFO] 3.txt -> matched_index=3 reason=numeric_suffix_matched_0based(3) -> Ga-Co.txt
[INFO] 4.txt -> matched_index=4 reason=numeric_suffix_matched_0based(4) -> W-Co.txt
[INFO] 5.txt -> matched_index=5 reason=numeric_suffix_matched_0based(5) -> Mn-Co.txt
[INFO] 6.txt -> matched_index=6 reason=numeric_suffix_matched_0based(6) -> Zn-Co.txt
[INFO] 7.txt -> matched_index=7 reason=numeric_suffix_matched_0based(7) -> Cd-Co.txt
[INFO] 8.txt -> matched_index=8 reason=numeric_suffix_matched_0based(8) -> La-Co.txt
[INFO] 9.txt -> matched_index=9 reason=numeric_suffix_matched_

In [ ]:
import os
import shutil
import json
import re
from collections import defaultdict

# ========== 配置区（按需修改） ==========
base_path = r"C:\Users\Admin\Documents\SynologyDrive\0-data\H-D_splitting\Raw Data\4-manual\robotic_arm\ML"
destination_path = r"C:\Users\Admin\Documents\GitHub\hd_separation_ml\data_extraction\all_data\CV20"
# ========================================

os.makedirs(destination_path, exist_ok=True)

def normalize(text):
    if text is None:
        return ""
    s = str(text).strip()
    # 去掉扩展名 / json 后缀，转换小写，统一一些符号
    s = re.sub(r'\.json$', '', s, flags=re.IGNORECASE)
    s = s.replace(' ', '_')
    s = s.lower()
    return s

def container_to_list(container):
    """
    把可能的 JSON 字段（list / dict / 单值）统一成 list。
    dict 会尝试按数字键排序（若键都是数字），否则按原 dict 顺序返回 values 列表。
    """
    if container is None:
        return []
    if isinstance(container, list):
        return container
    if isinstance(container, dict):
        # 尝试把数字键按数值排序
        keys = list(container.keys())
        try:
            # 若所有 key 可转 int，则按 int 排
            if all(re.fullmatch(r'-?\d+', str(k)) for k in keys):
                keys_sorted = sorted(keys, key=lambda k: int(k))
                return [container[k] for k in keys_sorted]
        except Exception:
            pass
        # fallback: values 顺序
        return list(container.values())
    # 单一值
    return [container]

def build_combos(data):
    s1 = container_to_list(data.get('substance1'))
    s2 = container_to_list(data.get('substance2'))
    n = max(len(s1), len(s2))
    # pad short lists with None
    s1 += [None] * (n - len(s1))
    s2 += [None] * (n - len(s2))
    combos = []
    for i in range(n):
        combos.append({
            'index': i,
            'sub1_raw': s1[i],
            'sub2_raw': s2[i],
            'sub1_norm': normalize(s1[i]),
            'sub2_norm': normalize(s2[i]),
            'label_raw': f"{s1[i]}-{s2[i]}" if s1[i] is not None or s2[i] is not None else None,
            'label_norm': normalize(f"{s1[i]}-{s2[i]}") if (s1[i] or s2[i]) else ""
        })
    return combos

def find_best_match_for_basename(basename_norm, basename_raw, combos):
    """
    尝试多策略匹配，返回匹配到的 index 或 None，以及匹配理由字符串
    策略顺序：
      1) 文件名尾部数字 -> 当作 index（支持 0-based 或 1-based 尝试）
      2) 精确 label match（sub1-sub2）
      3) 精确 sub1 或 sub2 匹配（若唯一则采用）
      4) 部分匹配（basename 包含在 label 中或 label 包含 basename）
      5) 无匹配 -> None
    """
    # 1) 尝试尾部数字
    m = re.search(r'(\d+)$', basename_raw)
    if m:
        num = int(m.group(1))
        # 尝试 0-based
        if 0 <= num < len(combos):
            return num, f"numeric_suffix_matched_0based({num})"
        # 尝试 1-based
        if 1 <= num <= len(combos):
            idx = num - 1
            return idx, f"numeric_suffix_matched_1based({num} -> {idx})"

    # 2) 精确 label match（sub1-sub2）
    for c in combos:
        if basename_norm and c['label_norm'] and basename_norm == c['label_norm']:
            return c['index'], "exact_label_match"

    # 3) 精确 sub1 或 sub2 唯一匹配
    sub1_matches = [c['index'] for c in combos if basename_norm and basename_norm == c['sub1_norm']]
    sub2_matches = [c['index'] for c in combos if basename_norm and basename_norm == c['sub2_norm']]
    if len(sub1_matches) == 1:
        return sub1_matches[0], "unique_sub1_match"
    if len(sub2_matches) == 1:
        return sub2_matches[0], "unique_sub2_match"

    # 4) 部分匹配（包含关系）
    partial = []
    for c in combos:
        if basename_norm and (basename_norm in c['label_norm'] or c['label_norm'] in basename_norm or basename_norm in c['sub1_norm'] or basename_norm in c['sub2_norm']):
            partial.append(c['index'])
    if len(partial) == 1:
        return partial[0], "partial_single_match"
    if len(partial) > 1:
        return partial[0], f"partial_multiple_match_choose_first({partial})"  # 可改为返回 None

    # 无匹配
    return None, "no_match"

# 主流程：遍历 base_path 下每个子文件夹
summary = []  # 用于写 mapping_summary.json
for folder in os.listdir(base_path):
    folder_path = os.path.join(base_path, folder)
    if not os.path.isdir(folder_path):
        continue

    tmp_json_path = os.path.join(folder_path, "tmp", "initial_sampling.json")
    if not os.path.exists(tmp_json_path):
        print(f"[WARN] JSON not found: {tmp_json_path}  (跳过)")
        continue

    try:
        with open(tmp_json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
    except Exception as e:
        print(f"[ERROR] Failed to load {tmp_json_path}: {e}")
        continue

    combos = build_combos(data)
    if not combos:
        print(f"[WARN] {tmp_json_path} 中没有 substance1/substance2 内容 (跳过)")
        continue

    EIS_path = os.path.join(folder_path, "tmp", "electrocatalysis", "CV","20")
    if not os.path.isdir(EIS_path):
        print(f"[WARN] EIS folder not found: {EIS_path} (跳过)")
        continue

    txt_files = sorted([f for f in os.listdir(EIS_path) if f.lower().endswith('.txt')])
    assigned_indices = {}   # index -> target info
    file_reports = []

    for fname in txt_files:
        base_raw = os.path.splitext(fname)[0]
        base_norm = normalize(base_raw)

        matched_idx, reason = find_best_match_for_basename(base_norm, base_raw, combos)

        if matched_idx is None:
            # 回退策略：按尚未填充的最小索引顺序分配
            unassigned = [c['index'] for c in combos if c['index'] not in assigned_indices]
            if unassigned:
                matched_idx = unassigned[0]
                reason = f"fallback_sequential_assigned({matched_idx})"
            else:
                reason = "fallback_no_unassigned_index"
                matched_idx = None

        target_name = None
        if matched_idx is not None:
            c = combos[matched_idx]
            s1 = c['sub1_raw'] if c['sub1_raw'] is not None else "UnknownSub1"
            s2 = c['sub2_raw'] if c['sub2_raw'] is not None else "UnknownSub2"
            # 生成安全的文件名（把可能的斜杠去掉）
            safe_s1 = re.sub(r'[\\/:*?"<>|]', '_', str(s1))
            safe_s2 = re.sub(r'[\\/:*?"<>|]', '_', str(s2))
            target_name = f"{safe_s1}-{safe_s2}.txt"
            # 若已存在同名（可能不同原文件匹配到同一 index），加上索引避免覆盖
            target_path = os.path.join(destination_path, target_name)
            if os.path.exists(target_path):
                target_name = f"{safe_s1}-{safe_s2}.txt"
            assigned_indices[matched_idx] = {
                'txt_source': os.path.join(EIS_path, fname),
                'target_name': target_name,
                'match_reason': reason
            }
            # 复制文件
            try:
                shutil.copy(os.path.join(EIS_path, fname), os.path.join(destination_path, target_name))
            except Exception as e:
                print(f"[ERROR] Copy failed for {fname}: {e}")
        else:
            # 无法匹配到任何 index，保留原名并在文件名加上 folder 前缀防止冲突
            target_name = f"{folder}_{base_raw}.txt"
            try:
                shutil.copy(os.path.join(EIS_path, fname), os.path.join(destination_path, target_name))
            except Exception as e:
                print(f"[ERROR] Copy fallback failed for {fname}: {e}")

        file_reports.append({
            'source_file': fname,
            'base_raw': base_raw,
            'base_norm': base_norm,
            'matched_index': matched_idx,
            'target_name': target_name,
            'reason': reason
        })
        print(f"[INFO] {fname} -> matched_index={matched_idx} reason={reason} -> {target_name}")

    # 记录哪些 index 没有被任何文件匹配到
    missing_indices = [c['index'] for c in combos if c['index'] not in assigned_indices]

    summary.append({
        'source_folder': folder,
        'json_path': tmp_json_path,
        'EIS_path': EIS_path,
        'num_json_entries': len(combos),
        'num_txt_files_found': len(txt_files),
        'assigned_count': len(assigned_indices),
        'missing_indices': missing_indices,
        'file_reports': file_reports
    })

# 写 summary 报告
summary_path = os.path.join(destination_path, "mapping_summary-1.json")
with open(summary_path, 'w', encoding='utf-8') as outf:
    json.dump(summary, outf, ensure_ascii=False, indent=2)

print(f"Done. Mapping summary saved to: {summary_path}")

[INFO] 0.txt -> matched_index=0 reason=numeric_suffix_matched_0based(0) -> Eu-Co.txt
[INFO] 1.txt -> matched_index=1 reason=numeric_suffix_matched_0based(1) -> Y-Co.txt
[INFO] 10.txt -> matched_index=10 reason=numeric_suffix_matched_0based(10) -> Cr-Co.txt
[INFO] 11.txt -> matched_index=11 reason=numeric_suffix_matched_0based(11) -> In-Co.txt
[INFO] 2.txt -> matched_index=2 reason=numeric_suffix_matched_0based(2) -> V-Co.txt
[INFO] 3.txt -> matched_index=3 reason=numeric_suffix_matched_0based(3) -> Ga-Co.txt
[INFO] 4.txt -> matched_index=4 reason=numeric_suffix_matched_0based(4) -> W-Co.txt
[INFO] 5.txt -> matched_index=5 reason=numeric_suffix_matched_0based(5) -> Mn-Co.txt
[INFO] 6.txt -> matched_index=6 reason=numeric_suffix_matched_0based(6) -> Zn-Co.txt
[INFO] 7.txt -> matched_index=7 reason=numeric_suffix_matched_0based(7) -> Cd-Co.txt
[INFO] 8.txt -> matched_index=8 reason=numeric_suffix_matched_0based(8) -> La-Co.txt
[INFO] 9.txt -> matched_index=9 reason=numeric_suffix_matched_

In [ ]:
import os

# 设置路径
cv20_path = r"C:\Users\Admin\Documents\GitHub\hd_separation_ml\data_extraction\all_data\CV20"
eis_path = r"C:\Users\Admin\Documents\GitHub\hd_separation_ml\data_extraction\all_data\EIS"

# 获取 CV20 文件夹中的文件
cv20_files = set(os.listdir(cv20_path))

# 获取 EIS 文件夹中的文件
eis_files = set(os.listdir(eis_path))

# 找出 EIS 中但不在 CV20 中的文件
missing_files = eis_files - cv20_files

# 打印缺失的文件
if missing_files:
    print("Files in EIS but missing in CV20:")
    for file in missing_files:
        print(file)
else:
    print("All files in EIS are present in CV20.")

All files in EIS are present in CV20.


In [ ]:
import os
import shutil
import json

# 设置路径
base_path = r"C:\Users\Admin\Documents\SynologyDrive\0-data\H-D_splitting\Raw Data\4-manual\robotic_arm\ML"
destination_path = r"C:\Users\Admin\Documents\GitHub\hd_separation_ml\data_extraction\all_data\Et"

# 创建目标文件夹（如果不存在）
os.makedirs(destination_path, exist_ok=True)

# 遍历每个子文件夹
for folder in os.listdir(base_path):
    folder_path = os.path.join(base_path, folder)
    if os.path.isdir(folder_path):
        print(f"Checking folder: {folder_path}")
        
        # 假设每个子文件夹都有一个 tmp 文件夹
        tmp_json_path = os.path.join(folder_path, "tmp", "initial_sampling.json")
        
        # 检查 JSON 文件是否存在
        if os.path.exists(tmp_json_path):
            with open(tmp_json_path, 'r') as f:
                data = json.load(f)

            # 查找 lsv 文件夹
            LSV_path = os.path.join(folder_path, "tmp", "electrocatalysis", "Et")
            if os.path.exists(LSV_path):
                # 复制并重命名文件
                for i, txt_file in enumerate(os.listdir(LSV_path)):
                    if txt_file.endswith('.txt'):
                        # 生成新文件名
                        substance1 = data['substance1'].get(str(i), "UnknownSubstance1")
                        substance2 = data['substance2'].get(str(i), "UnknownSubstance2")
                        new_name = f"{substance1}-{substance2}.txt"

                        # 检查重命名是否成功
                        if substance1 == "UnknownSubstance1" or substance2 == "UnknownSubstance2":
                            print(f"Warning: No valid substance for index {i} in {tmp_json_path}")

                        new_file_path = os.path.join(destination_path, new_name)
                        original_file_path = os.path.join(LSV_path, txt_file)
                        shutil.copy(original_file_path, new_file_path)
                        print(f"Copied and renamed: {original_file_path} to {new_file_path}")
                        
                    elif txt_file.endswith('.png'):
                        print(f"Skipping PNG file: {txt_file}")
            else:
                print(f"LSV folder not found in: {folder_path}")
        else:
            print(f"JSON file not found in: {folder_path}")

Checking folder: C:\Users\Admin\Documents\SynologyDrive\0-data\H-D_splitting\Raw Data\4-manual\robotic_arm\ML\0-Co
Copied and renamed: C:\Users\Admin\Documents\SynologyDrive\0-data\H-D_splitting\Raw Data\4-manual\robotic_arm\ML\0-Co\tmp\electrocatalysis\Et\0.txt to C:\Users\Admin\Documents\SynologyDrive\0-data\H-D_splitting\Raw Data\4-manual\robotic_arm\ML\ML-20250926\Et\Eu-Co.txt
Copied and renamed: C:\Users\Admin\Documents\SynologyDrive\0-data\H-D_splitting\Raw Data\4-manual\robotic_arm\ML\0-Co\tmp\electrocatalysis\Et\1.txt to C:\Users\Admin\Documents\SynologyDrive\0-data\H-D_splitting\Raw Data\4-manual\robotic_arm\ML\ML-20250926\Et\Y-Co.txt
Copied and renamed: C:\Users\Admin\Documents\SynologyDrive\0-data\H-D_splitting\Raw Data\4-manual\robotic_arm\ML\0-Co\tmp\electrocatalysis\Et\10.txt to C:\Users\Admin\Documents\SynologyDrive\0-data\H-D_splitting\Raw Data\4-manual\robotic_arm\ML\ML-20250926\Et\V-Co.txt
Copied and renamed: C:\Users\Admin\Documents\SynologyDrive\0-data\H-D_splitting

In [ ]:
import os
import json

base_path = r"C:\Users\Admin\Documents\SynologyDrive\0-data\H-D_splitting\Raw Data\4-manual\robotic_arm\ML"
destination_path = r"C:\Users\Admin\Documents\GitHub\hd_separation_ml\data_extraction\all_data"

os.makedirs(destination_path, exist_ok=True)

output_list = []

def to_float(val):
    """把可能的数值（字符串/数字）安全转成 float，失败返回 None"""
    if val is None:
        return None
    # 如果是 list/dict 等不可直接转型，直接跳过
    if isinstance(val, (list, dict)):
        return None
    try:
        # 去掉逗号并替换中文逗号，去除空白
        if isinstance(val, str):
            v = val.strip().replace('，', ',')
            # 若字符串里包含逗号（千分位），移除
            v = v.replace(',', '')
        else:
            v = val
        return float(v)
    except Exception:
        return None

def extract_by_index(container, i):
    """
    从 container（可能是 dict/list/单值）中提取索引 i 的对应值。
    支持：
      - 如果 container 是 dict，则尝试 str(i), i, 0（fallback）
      - 如果 container 是 list/tuple，则用索引 i（若越界返回 None）
      - 否则直接返回 container（当 container 为单个值时适用）
    """
    if container is None:
        return None
    if isinstance(container, dict):
        # 尝试几种键形式
        for key in (str(i), i, 0):
            if key in container:
                return container[key]
        # 还可以尝试返回第一个可用值作为回退
        if len(container) > 0:
            # 取第一个 value
            first_key = next(iter(container))
            return container[first_key]
        return None
    if isinstance(container, (list, tuple)):
        if 0 <= i < len(container):
            return container[i]
        else:
            return None
    # 单个值
    return container

# 遍历子文件夹
for folder in os.listdir(base_path):
    folder_path = os.path.join(base_path, folder)
    if not os.path.isdir(folder_path):
        continue

    tmp_json_path = os.path.join(folder_path, "tmp", "initial_sampling.json")
    if not os.path.exists(tmp_json_path):
        print(f"JSON not found: {tmp_json_path}")
        continue

    try:
        with open(tmp_json_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
    except Exception as e:
        print(f"Failed to load {tmp_json_path}: {e}")
        continue

    # 预期可能包含的字段名（注意原始 key 名中含 '-' 的字段）
    key_sub1 = 'substance1'
    key_sub2 = 'substance2'
    key_ir = 'ir_value'
    key_lsv = 'lsv_overpotential_-0.01'  # 和你要求的名字一致

    sub1_container = data.get(key_sub1)
    sub2_container = data.get(key_sub2)
    ir_container = data.get(key_ir)
    lsv_container = data.get(key_lsv)

    # 如果以上字段都不存在，尝试宽松查找（避免因为命名差异漏掉）
    if sub1_container is None and 'substance' in data:
        sub1_container = data.get('substance')  # 作为备选
    # 遍历文件夹下的 LSV 文件数作为索引上限（如果需要与原脚本保持一致）
    # 先尝试找 LSV 目录下的 txt 文件数量，如果没有 LSV 文件夹，仍然尝试使用 max len of containers
    LSV_path = os.path.join(folder_path, "tmp", "electrocatalysis", "LSV")
    file_count = 0
    if os.path.isdir(LSV_path):
        try:
            file_count = len([f for f in os.listdir(LSV_path) if f.lower().endswith('.txt')])
        except Exception:
            file_count = 0
    

    # 作为保险，如果 file_count 为 0，尝试用 containers 的最大长度
    candidates_lengths = []
    for c in (sub1_container, sub2_container, ir_container, lsv_container):
        if isinstance(c, (list, tuple)):
            candidates_lengths.append(len(c))
        elif isinstance(c, dict):
            candidates_lengths.append(len(c))
    if file_count == 0 and candidates_lengths:
        file_count = max(candidates_lengths)

    if file_count == 0:
        # 仍然无法推断数量，尝试把该 json 当作单条记录处理
        file_count = 1

    for i in range(file_count):
        raw_sub1 = extract_by_index(sub1_container, i)
        raw_sub2 = extract_by_index(sub2_container, i)
        raw_ir = extract_by_index(ir_container, i)
        raw_lsv = extract_by_index(lsv_container, i)

        # 如果容器是 None，但 data 中存在直接键为 'substance1_0' 之类的，尝试匹配（宽松）
        if raw_sub1 is None:
            # 搜索类似的键
            for k in data.keys():
                if k.startswith('substance1') and (k.endswith(str(i)) or k.endswith(f"_{i}")):
                    raw_sub1 = data[k]
                    break
        if raw_sub2 is None:
            for k in data.keys():
                if k.startswith('substance2') and (k.endswith(str(i)) or k.endswith(f"_{i}")):
                    raw_sub2 = data[k]
                    break
        if raw_ir is None:
            for k in data.keys():
                if 'ir' in k.lower() and (k.endswith(str(i)) or k.endswith(f"_{i}") or k == key_ir):
                    raw_ir = data[k]
                    break
        if raw_lsv is None:
            for k in data.keys():
                if 'lsv' in k.lower() and (k.endswith(str(i)) or k.endswith(f"_{i}") or k == key_lsv):
                    raw_lsv = data[k]
                    break

        ir_val = to_float(raw_ir)
        lsv_val = to_float(raw_lsv)

        if ir_val is None or lsv_val is None:
            print(f"Skipping entry: folder={folder}, index={i} due to missing numeric ir or lsv. ir_raw={raw_ir}, lsv_raw={raw_lsv}")
            continue

        EIS_V = lsv_val - ir_val * 0.01

        entry = {
        
            "substance1": raw_sub1 if raw_sub1 is not None else "UnknownSubstance1",
            "substance2": raw_sub2 if raw_sub2 is not None else "UnknownSubstance2",
            "ir_value": ir_val,
            "lsv_overpotential_-0.01": lsv_val,
            "EIS_V": EIS_V
        }
        output_list.append(entry)
        print(f"Added: sub2={entry['substance2']} ir={ir_val} lsv={lsv_val} EIS_V={EIS_V}")

# 最终写入合并 JSON 文件
out_path = os.path.join(destination_path, "all_substances.json")
try:
    with open(out_path, 'w', encoding='utf-8') as outf:
        json.dump(output_list, outf, ensure_ascii=False, indent=2)
    print(f"Saved {len(output_list)} records to {out_path}")
except Exception as e:
    print(f"Failed to write output JSON: {e}")



Added: sub2=Co ir=3.67 lsv=-1.1548341854 EIS_V=-1.1915341854
Added: sub2=Co ir=3.75 lsv=-1.1494689238 EIS_V=-1.1869689238
Added: sub2=Co ir=3.6883895795 lsv=-1.0432161566 EIS_V=-1.080100052395
Added: sub2=Co ir=3.6870106856 lsv=-1.0901246156 EIS_V=-1.126994722456
Added: sub2=Co ir=3.7411499023 lsv=-1.0352268678 EIS_V=-1.072638366823
Added: sub2=Co ir=4.0576146444 lsv=-1.185807466 EIS_V=-1.226383612444
Added: sub2=Co ir=4.1878995895 lsv=-1.291021332 EIS_V=-1.332900327895
Added: sub2=Co ir=4.1709809303 lsv=-1.2872274974 EIS_V=-1.328937306703
Added: sub2=Co ir=4.0715248585 lsv=-1.3895781913 EIS_V=-1.430293439885
Added: sub2=Co ir=4.4448525906 lsv=-1.4091607626 EIS_V=-1.453609288506
Added: sub2=Co ir=4.7141545614 lsv=-1.4735779991 EIS_V=-1.520719544714
Added: sub2=Co ir=4.7145814896 lsv=-1.391880015 EIS_V=-1.439025829896
Added: sub2=Nb ir=4.03 lsv=-1.1374581331 EIS_V=-1.1777581331
Added: sub2=Nb ir=3.62 lsv=-1.1981244087 EIS_V=-1.2343244087
Added: sub2=Nb ir=3.98 lsv=-1.1410372388 EIS_V=-1

In [ ]:
import os
import csv

# ========== 配置区（按需修改） ==========
ET_FOLDER = r"C:\Users\Admin\Documents\GitHub\hd_separation_ml\data_extraction\all_data\CV20"
CSV_PATH = r"C:\Users\Admin\Documents\GitHub\hd_separation_ml\data_extraction\all_data\extracted_separation_coefficients.csv"
OUTPUT_CSV = os.path.join(os.path.dirname(CSV_PATH), "missing_in_Et.csv")
SAMPLE_COL_NAME = "sample"   # CSV 中的列名
CASE_INSENSITIVE = True      # True -> 比较时忽略大小写；False -> 区分大小写
# ========================================

def normalize_name(name: str) -> str:
    """去掉空白、两侧引号，去掉 .json 后缀并根据 CASE_INSENSITIVE 进行大小写规范化"""
    if name is None:
        return ""
    s = str(name).strip().strip('"').strip("'")
    if s.lower().endswith(".json"):
        s = s[: -5]   # 去掉 .json
    return s.lower() if CASE_INSENSITIVE else s

# 1) 收集 Et 文件夹内所有 .txt 文件（去掉扩展名）
txt_names = set()
if not os.path.isdir(ET_FOLDER):
    raise SystemExit(f"Et 文件夹不存在：{ET_FOLDER}")

for fname in os.listdir(ET_FOLDER):
    if fname.lower().endswith(".txt"):
        base = fname[:-4]  # 去掉 .txt
        txt_names.add(base.lower() if CASE_INSENSITIVE else base)

print(f"Found {len(txt_names)} .txt files in Et folder.")

# 2) 读取 CSV 中 sample 列
csv_samples = []
if not os.path.exists(CSV_PATH):
    raise SystemExit(f"CSV 文件不存在：{CSV_PATH}")

with open(CSV_PATH, newline='', encoding='utf-8') as csvfile:
    reader = csv.DictReader(csvfile)
    if SAMPLE_COL_NAME not in reader.fieldnames:
        raise SystemExit(f"CSV 中未找到列 '{SAMPLE_COL_NAME}'。可用列名：{reader.fieldnames}")
    for row in reader:
        raw = row.get(SAMPLE_COL_NAME)
        norm = normalize_name(raw)
        if norm:  # 忽略空串
            csv_samples.append(norm)

csv_samples_set = set(csv_samples)
print(f"Found {len(csv_samples_set)} unique samples in CSV (column '{SAMPLE_COL_NAME}').")

# 3) 比较：CSV 中有但 Et 文件夹没有的名称
missing = sorted(list(csv_samples_set - txt_names))

print(f"Missing count: {len(missing)}")

# 4) 写入输出 CSV
with open(OUTPUT_CSV, 'w', newline='', encoding='utf-8') as out:
    writer = csv.writer(out)
    writer.writerow([f"{SAMPLE_COL_NAME}_missing"])
    for name in missing:
        writer.writerow([name])

print(f"Saved missing list to: {OUTPUT_CSV}")

# 5) 简短输出（也方便直接查看）
if missing:
    print("示例缺失项（前 20 个，如果很多只列出前 20）：")
    for i, n in enumerate(missing[:20], start=1):
        print(f"{i}. {n}")
else:
    print("没有缺失：CSV 中的所有样本在 Et 文件夹的 .txt 中都找到了。")


Found 698 .txt files in Et folder.
Found 699 unique samples in CSV (column 'sample').
Missing count: 1
Saved missing list to: C:\Users\Admin\Documents\SynologyDrive\0-data\H-D_splitting\Raw Data\4-manual\robotic_arm\ML\ML-20250926\missing_in_Et.csv
示例缺失项（前 20 个，如果很多只列出前 20）：
1. co-ni4


In [ ]:
# calc_et_means.py
import os
import pandas as pd
import numpy as np

# ====== 配置区：修改为你的 Et 文件夹路径 ======
ET_DIR = r"C:\Users\Admin\Documents\GitHub\hd_separation_ml\data_extraction\all_data\Et"
OUTPUT_XLSX = os.path.join(os.path.dirname(ET_DIR), "Et_potential_means_1100_1200.xlsx")
TIME_MIN = 1100.0
TIME_MAX = 1200.0
# ==============================================

def read_txt_table(path):
    """
    读取以空白/制表符/逗号分隔的表格，返回 DataFrame。
    该项目的 txt 文件样例头部为:
    Potential(V)    Current(A)    Time(s)
    列名会被原样读取（含括号）。
    """
    # 使用正则分隔（空白/制表符/逗号），兼容各种分隔符
    return pd.read_csv(path, sep=r"\s+|,|\t", engine="python", comment='#', encoding='utf-8')

def find_col(cols, candidates):
    """在 cols 列名列表中查找首个匹配 candidates 的列（匹配包含，case-insensitive）"""
    lower = [c.lower() for c in cols]
    for cand in candidates:
        for i, c in enumerate(lower):
            if cand.lower() == c:
                return cols[i]
    for cand in candidates:
        for i, c in enumerate(lower):
            if cand.lower() in c:
                return cols[i]
    return None

def compute_mean_for_file(path):
    try:
        df = read_txt_table(path)
    except Exception as e:
        return np.nan, f"read_error: {e}"

    cols = [str(c).strip() for c in df.columns]

    # 优先用精确列名，然后用包含匹配
    time_candidates = ["Time(s)", "Time", "time(s)", "time"]
    pot_candidates  = ["Potential(V)", "Potential", "potential(v)", "potential"]

    time_col = find_col(cols, time_candidates)
    pot_col  = find_col(cols, pot_candidates)

    if time_col is None or pot_col is None:
        return np.nan, f"missing_columns: found {cols[:10]}"

    # 转为数值
    df[time_col] = pd.to_numeric(df[time_col], errors='coerce')
    df[pot_col]  = pd.to_numeric(df[pot_col], errors='coerce')

    # 取 1100-1200 窗口 (包含端点)
    window = df[(df[time_col] >= TIME_MIN) & (df[time_col] <= TIME_MAX)][pot_col].dropna()

    if window.size == 0:
        return np.nan, "no_data_in_time_window"

    return float(window.mean()), None

def main():
    if not os.path.isdir(ET_DIR):
        print(f"ERROR: Et folder not found: {ET_DIR}")
        return

    txt_files = sorted([f for f in os.listdir(ET_DIR) if f.lower().endswith(".txt")])
    print(f"Found {len(txt_files)} .txt files in {ET_DIR}")

    rows = []
    for fn in txt_files:
        full = os.path.join(ET_DIR, fn)
        basename = os.path.splitext(fn)[0]
        mean_val, err = compute_mean_for_file(full)
        rows.append({
            "filename": basename,
            "potential_mean_1100_1200": mean_val,
            "error": err
        })
        print(f"{fn} -> mean={mean_val} error={err}")

    df_out = pd.DataFrame(rows, columns=["filename", "potential_mean_1100_1200", "error"])
    # 将结果写入 Excel（第一列：文件名，第二列：平均值；保留 error 便于排查）
    df_out.to_excel(OUTPUT_XLSX, index=False)
    print(f"Saved results to: {OUTPUT_XLSX}")

if __name__ == "__main__":
    main()


Found 698 .txt files in C:\Users\Admin\Documents\SynologyDrive\0-data\H-D_splitting\Raw Data\4-manual\robotic_arm\ML\ML-20250926\Et
Ag-Ag.txt -> mean=-1.8328897769817574 error=None
Ag-Al.txt -> mean=-1.8810739153159592 error=None
Ag-Au.txt -> mean=-1.7940425218460327 error=None
Ag-Ba.txt -> mean=-1.714958945910136 error=None
Ag-Ca.txt -> mean=-1.7480987926681122 error=None
Ag-Cd.txt -> mean=-1.9234590518498373 error=None
Ag-Co.txt -> mean=-1.7451070049803652 error=None
Ag-Cr.txt -> mean=-1.6514169394137141 error=None
Ag-Cu.txt -> mean=-1.7533445265478718 error=None
Ag-Fe.txt -> mean=-1.6183633373645014 error=None
Ag-Ga.txt -> mean=-2.0269900600829285 error=None
Ag-In.txt -> mean=-2.103974043490168 error=None
Ag-La.txt -> mean=-1.9726547885559753 error=None
Ag-Mg.txt -> mean=-1.6978867170577516 error=None
Ag-Mn.txt -> mean=-1.8267046217908878 error=None
Ag-Mo.txt -> mean=-1.7844206542074086 error=None
Ag-Nb.txt -> mean=-1.5852445449657784 error=None
Ag-Ni.txt -> mean=-1.4799009887520187